# Case Study 1: Customer Service Chatbot for Indigo Airlines

In this notebook we build a multi-agent customer service chatbot for Indigo Airlines using the OpenAI Agents SDK — introducing each concept step by step before putting the full system together.

| Section | Concept |
|---|---|
| 1 | Basic agent — no tools, plain text output |
| 2 | Tools — hosted (WebSearch, FileSearch) and custom function tools |
| 3 | Handoffs — routing between specialist agents |
| 4 | Context — shared state across agents |
| 5 | Guardrails — input and output safety checks |

In [ ]:
# !pip install -q openai-agents python-dotenv

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner

load_dotenv("../../.env")

## Section 1: A Basic Agent

In [ ]:
basic_agent = Agent(
    name="CustomerServiceAgent",
    instructions="You are a friendly Indigo Airlines customer service agent. Answer customer questions clearly and concisely.",
    model="gpt-4o-mini",
)

In [ ]:
result = await Runner.run(basic_agent, "What is your refund policy?")
print(result.final_output)

In [ ]:
result = await Runner.run(basic_agent, "What is the wifi name on the flight?")
print(result.final_output)

## Section 2: Tools

Tools let an agent interact with the outside world. There are two kinds:

- **Hosted tools** — run on OpenAI's servers. You just pass them in; no code to write. Examples: `WebSearchTool`, `FileSearchTool`.
- **Custom tools** — plain Python functions you decorate with `@function_tool`. The SDK reads the name, docstring, and type hints to describe the tool to the model.

### `WebSearchTool` — Live Web Search

The simplest hosted tool. The model decides when to search and what to search for — you just attach it to the agent.

In [ ]:
from agents import WebSearchTool

In [ ]:
web_agent = Agent(
    name="IndiGo Web Agent",
    instructions=(
        "You are an IndiGo Airlines customer service agent. "
        "Use web search to find up-to-date information before answering. "
        "Always mention what you found from the web."
    ),
    model="gpt-4o-mini",
    tools=[WebSearchTool()],
)

result = await Runner.run(web_agent, "What are IndiGo's wifi name in domestic flights in 2026?")
print(result.final_output)

In [ ]:
result = await Runner.run(web_agent, "What new routes has IndiGo launched recently?")
print(result.final_output)

### `@function_tool` — Custom Python Function as a Tool

`WebSearchTool` is convenient, but sometimes you need the agent to call your own backend logic — a database lookup, a pricing API, or a simple rule engine. That's where `@function_tool` comes in.

First, let's write the lookup as a plain Python function and call it directly to understand what it returns.

In [ ]:
def faq_lookup(question: str) -> str:
    question_lower = question.lower()

    if any(word in question_lower for word in ["bag", "baggage", "luggage", "carry-on"]):
        return (
            "You are allowed one carry-on bag. "
            "It must be under 7 kgs and fit in the overhead bin (22x14x9 inches)."
        )

    if any(word in question_lower for word in ["seat", "seats", "seating"]):
        return (
            "The plane has 120 seats: 22 business class and 98 economy. "
            "Rows 5-8 are Economy Plus with extra legroom."
        )

    if any(word in question_lower for word in ["wifi", "internet", "wireless"]):
        return "Wifi is unavailable on the flight."

    return "I'm sorry, I don't have an answer for that. Please contact support."

In [ ]:
print(faq_lookup("How heavy can my carry-on bag be?"))
# print(faq_lookup("Is there wifi on the flight?"))
# print(faq_lookup("What seats are available?"))
# print(faq_lookup("Can I bring my pet?"))
# print(faq_lookup("Can my carry on bag be 10 kgs"))

### Turning it into an Agent Tool

In [ ]:
import json
from agents import function_tool

In [ ]:
@function_tool
def faq_lookup_tool(question: str) -> str:
    """Look up answers to frequently asked questions about the airline."""
    question_lower = question.lower()

    if any(word in question_lower for word in ["bag", "baggage", "luggage", "carry-on"]):
        return (
            "You are allowed one carry-on bag. "
            "It must be under 7 kgs and fit in the overhead bin (22x14x9 inches)."
        )

    if any(word in question_lower for word in ["seat", "seats", "seating"]):
        return (
            "The plane has 120 seats: 22 business class and 98 economy. "
            "Rows 5-8 are Economy Plus with extra legroom."
        )

    if any(word in question_lower for word in ["wifi", "internet", "wireless"]):
        return "Wifi is unavailable on the flight.'."

    return "I'm sorry, I don't have an answer for that. Please contact support."

In [ ]:
# The decorator turns our function into a FunctionTool object.
# This is the exact schema sent to the model — it uses it to decide when and how to call the tool.
print("\n Name        :", faq_lookup_tool.name)
print("\n Description :", faq_lookup_tool.description)
print("\n Parameters  :")
print(json.dumps(faq_lookup_tool.params_json_schema, indent=2))

In [ ]:
faq_agent = Agent(
    name="FAQAgent",
    instructions=(
        "You are a helpful Indigo Airlines Customer Service agent. "
        "Always use the faq_lookup_tool to answer questions — never guess from memory."
    ),
    model="gpt-4o-mini",
    tools=[faq_lookup_tool],
)

result = await Runner.run(faq_agent, "Can my carry on bag be 10 kgs?")
print(result.final_output)

In [ ]:
result = await Runner.run(faq_agent, "How do I connect my phone to the internet on the flight?")
print(result.final_output)

### `FileSearchTool` — Searching a Real Policy Document

In [ ]:
# The PDF lives in the docs folder — real IndiGo domestic FAQ content.
FAQ_PDF_PATH = "../docs/IndiGo_FAQ.pdf"
print("Using:", FAQ_PDF_PATH)

In [ ]:
import time
from openai import OpenAI
from agents import FileSearchTool

In [ ]:
client = OpenAI()

# 1. Upload the PDF
with open(FAQ_PDF_PATH, "rb") as f:
    uploaded_file = client.files.create(file=f, purpose="assistants")

# 2. Create a vector store and add the file
vector_store = client.vector_stores.create(name="IndiGo FAQ")

client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=uploaded_file.id,
)

# 3. Wait for indexing to complete
while True:
    files = client.vector_stores.files.list(vector_store_id=vector_store.id).data
    if files and files[0].status == "completed":
        break
    time.sleep(1)

print("Vector store ready:", vector_store.id)

In [ ]:
faq_pdf_agent = Agent(
    name="IndiGo FAQ Agent",
    instructions=(
        "You are an IndiGo Airlines customer service agent. "
        "Always use the file search tool to answer questions — never rely on your own knowledge."
    ),
    model="gpt-4o-mini",
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
)

result = await Runner.run(faq_pdf_agent, "What is the free baggage allowance on domestic flights?")
print(result.final_output)


In [ ]:
result = await Runner.run(faq_pdf_agent, "There are 100 hrs to my flight. How can I web check in?")
print(result.final_output)

### Forcing Tool Use with `tool_choice`

You can override this with `ModelSettings(tool_choice=...)`:

| Value | Behaviour |
|---|---|
| `"auto"` | Model decides (default) |
| `"required"` | Model **must** call at least one tool before replying |
| `"none"` | Model may not call any tools |

In [ ]:
from agents import ModelSettings

In [ ]:
strict_faq_agent = Agent(
    name="IndiGo Strict FAQ Agent",
    instructions=(
        "You are an IndiGo Airlines customer service agent. "
        "Only answer from the policy document. If the document does not contain the answer, say so."
    ),
    model="gpt-4o-mini",
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
    model_settings=ModelSettings(tool_choice="required"),  # always search before replying
)

In [ ]:
result = await Runner.run(strict_faq_agent, "How much does it cost to change a Flexi Plus fare flight?")
print(result.final_output)

In [ ]:
result = await Runner.run(strict_faq_agent, "What happens if my bag is lost on a domestic flight?")
print(result.final_output)

## Section 3: Multiple Agents + Handoffs

A single agent can handle anything, but a better pattern for customer service is to have **specialist agents** — each focused on one task — with a **triage agent** that routes the customer to the right one.

The `handoff()` function tells an agent it can transfer the conversation to another agent. The model reads the `handoff_description` of each target agent to decide where to route.

You can inspect `result.new_items` to see exactly what happened — messages, tool calls, and handoffs.

In [ ]:
from agents import HandoffOutputItem, ItemHelpers, MessageOutputItem, handoff

In [ ]:
@function_tool
def update_seat(confirmation_number: str, new_seat: str) -> str:
    """Update the seat for a given booking confirmation number."""
    return f"Seat updated to {new_seat} for confirmation {confirmation_number}."


# --- Specialist agents ---

faq_specialist = Agent(
    name="FAQ Specialist",
    handoff_description="Answers general questions about baggage, seats, wifi, and policies.",
    instructions=(
        "You are an IndiGo Airlines customer service agent. "
        "Always use the file search tool to answer questions — never rely on your own knowledge."
    ),
    model="gpt-4o-mini",
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
    model_settings=ModelSettings(tool_choice="required"),
)

seat_specialist = Agent(
    name="Seat Booking Specialist",
    handoff_description="Handles seat changes and booking updates.",
    instructions=(
        "You are a seat booking agent for Indigo Airlines. "
        "Use whatever confirmation number and seat the customer has already provided. "
        "Only ask for information that is missing, then call update_seat."
    ),
    model="gpt-4o-mini",
    tools=[update_seat],
    model_settings=ModelSettings(tool_choice="required")
)

# --- Triage agent ---

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You are the first point of contact for customer service for Indigo Airlines. "
        "Route the customer to the right specialist agent based on their request."
    ),
    model="gpt-4o-mini",
    handoffs=[
        handoff(faq_specialist),
        handoff(seat_specialist),
    ],
)

print("Agents ready.")

In [ ]:
# Run a question that should be routed to the FAQ specialist
result = await Runner.run(triage_agent, "How much can my carry-on bag weigh?")

for item in result.new_items:
    if isinstance(item, HandoffOutputItem):
        print(f"[Handoff] {item.source_agent.name} → {item.target_agent.name}")
    elif isinstance(item, MessageOutputItem):
        print(f"[{item.agent.name}] {ItemHelpers.text_message_output(item)}")

In [ ]:
# Run a request that should be routed to the seat booking specialist
result = await Runner.run(triage_agent, "I'd like to change my seat to 12A. My confirmation is ABC123.")

for item in result.new_items:
    if isinstance(item, HandoffOutputItem):
        print(f"[Handoff] {item.source_agent.name} → {item.target_agent.name}")
    elif isinstance(item, MessageOutputItem):
        print(f"[{item.agent.name}] {ItemHelpers.text_message_output(item)}")

## Section 4: Shared Context (State)

So far each run is stateless — agents don't remember anything between calls. For real customer service you often need to carry **state** across agents and turns, like the customer's name or booking details.

The SDK supports this with a **context** object — a plain Pydantic model you create once and pass into `Runner.run()`. Any agent or tool can read and write it via `RunContextWrapper`.

When a tool accepts `context: RunContextWrapper[YourModel]` as its first argument, the SDK injects it automatically — the model never sees it.

In [ ]:
from pydantic import BaseModel
from agents import RunContextWrapper

In [ ]:
# --- Context model ---

class BookingContext(BaseModel):
    passenger_name: str | None = None
    confirmation_number: str | None = None
    seat_number: str | None = None


# --- Tools ---

@function_tool
def update_seat(
    context: RunContextWrapper[BookingContext],
    confirmation_number: str,
    new_seat: str,
) -> str:
    """Update the seat for a given booking confirmation number."""
    context.context.confirmation_number = confirmation_number
    context.context.seat_number = new_seat
    return f"Seat updated to {new_seat} for confirmation {confirmation_number}."


# --- Agents typed with BookingContext ---

faq_specialist = Agent[BookingContext](
    name="FAQ Specialist",
    handoff_description="Answers general questions about baggage, seats, wifi, and policies.",
    instructions=(
        "You are an IndiGo Airlines customer service agent. "
        "Always use the file search tool to answer questions — never rely on your own knowledge."
    ),
    model="gpt-4o-mini",
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
)

seat_specialist = Agent[BookingContext](
    name="Seat Booking Specialist",
    handoff_description="Handles seat changes and booking updates.",
    instructions=(
        "You are a seat booking agent. "
        "Use whatever confirmation number and seat the customer has already provided. "
        "Only ask for information that is missing, then call update_seat."
    ),
    model="gpt-4o-mini",
    tools=[update_seat],
)

triage_agent = Agent[BookingContext](
    name="Triage Agent",
    instructions=(
        "You are the first point of contact for customer service. "
        "Route the customer to the right specialist agent based on their request."
    ),
    model="gpt-4o-mini",
    handoffs=[
        handoff(faq_specialist),
        handoff(seat_specialist),
    ],
)

print("Agents with context ready.")

In [ ]:
# Create a context object for this customer session
ctx = BookingContext(passenger_name="Alice")

print("\n--- Context before run ---")
print(f"Passenger : {ctx.passenger_name}")
print(f"Confirmation: {ctx.confirmation_number}")
print(f"Seat        : {ctx.seat_number}")

In [ ]:
result = await Runner.run(
    triage_agent,
    "Please change my seat to 14C. My confirmation number is XYZ789.",
    context=ctx,
)

In [ ]:
for item in result.new_items:
    if isinstance(item, HandoffOutputItem):
        print(f"[Handoff] {item.source_agent.name} → {item.target_agent.name}")
    elif isinstance(item, MessageOutputItem):
        print(f"[{item.agent.name}] {ItemHelpers.text_message_output(item)}")

In [ ]:
# Inspect the context after the run — seat and confirmation were saved
print("\n--- Context after run ---")
print(f"Passenger : {ctx.passenger_name}")
print(f"Confirmation: {ctx.confirmation_number}")
print(f"Seat        : {ctx.seat_number}")

In [ ]:
# Second example: a new session, route to FAQ specialist, context stays clean
ctx2 = BookingContext(passenger_name="Rahul")

print("\n--- Context before run ---")
print(f"Passenger : {ctx2.passenger_name}")
print(f"Confirmation: {ctx2.confirmation_number}")
print(f"Seat        : {ctx2.seat_number}")


In [ ]:
result = await Runner.run(
    triage_agent,
    "I want to add 10 kg of extra baggage to my flight. How do I do that?",
    context=ctx2,
)

for item in result.new_items:
    if isinstance(item, HandoffOutputItem):
        print(f"[Handoff] {item.source_agent.name} → {item.target_agent.name}")
    elif isinstance(item, MessageOutputItem):
        print(f"[{item.agent.name}] {ItemHelpers.text_message_output(item)}")

print("\n--- Context after run ---")
print(f"Passenger : {ctx2.passenger_name}")
print(f"Confirmation: {ctx2.confirmation_number}")
print(f"Seat        : {ctx2.seat_number}")

## Section 5: Guardrails

Guardrails let you add safety checks **around** an agent without changing its logic.

- **Input guardrail** — runs *before* the agent processes the user message. Use it to reject off-topic or abusive queries.
- **Output guardrail** — runs *after* the agent produces a response. Use it to catch hallucinations or policy violations.

You define a guardrail as a function decorated with `@input_guardrail` or `@output_guardrail`. It returns a `GuardrailFunctionOutput` with `tripwire_triggered=True` to block, or `False` to let through. When a guardrail trips, the SDK raises `InputGuardrailTripwireTriggered` or `OutputGuardrailTripwireTriggered`.

Guardrails are attached to an agent via `input_guardrails=[...]` and `output_guardrails=[...]`.

### Input Guardrail — Block Off-Topic Questions

The chatbot should only answer airline-related questions. If a customer sends something completely off-topic (cricket scores, recipes, stock prices), we want to reject it immediately without wasting a model call on the main agent.

In [ ]:
from agents import (
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
    input_guardrail,
    output_guardrail,
)

In [ ]:
OFF_TOPIC_KEYWORDS = [
    "cricket", "recipe", "stock market", "movie", "politics",
    "weather", "bollywood", "ipl", "football",
]

@input_guardrail
async def airline_topic_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list,
) -> GuardrailFunctionOutput:
    """Block messages that are clearly not related to airline travel."""
    text = str(input).lower()
    is_off_topic = any(kw in text for kw in OFF_TOPIC_KEYWORDS)
    return GuardrailFunctionOutput(
        output_info=f"Off-topic query blocked: {text[:60]}",
        tripwire_triggered=is_off_topic,
    )

guarded_agent = Agent(
    name="Guarded IndiGo Agent",
    instructions="You are an IndiGo Airlines customer service agent. Answer only airline-related questions.",
    model="gpt-4o-mini",
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
    input_guardrails=[airline_topic_guardrail],
)

print("Guarded agent ready.")

In [ ]:
# Example 1: valid airline question — guardrail passes, agent answers normally
try:
    result = await Runner.run(guarded_agent, "What is IndiGo's web check-in policy?")
    print("ALLOWED:", result.final_output)
except InputGuardrailTripwireTriggered as e:
    print("BLOCKED by input guardrail:", e)

In [ ]:
# Example 2: off-topic question — guardrail trips before the agent even runs
try:
    result = await Runner.run(guarded_agent, "Who won the IPL this year?")
    print("ALLOWED:", result.final_output)
except InputGuardrailTripwireTriggered as e:
    print("BLOCKED by input guardrail — off-topic question detected.")

In [ ]:
# Example 3: another valid question — guardrail passes
try:
    result = await Runner.run(guarded_agent, "How is the weather in Mumbai?")
    print("ALLOWED:", result.final_output)
except InputGuardrailTripwireTriggered as e:
    print("BLOCKED by input guardrail:", e)

### Output Guardrail — Catch Problematic Responses

Input guardrails protect the *entry* point. Output guardrails protect the *exit* point — they inspect what the agent produced before it reaches the customer.

Here we add a simple quality check: if the agent's response is suspiciously short (fewer than 20 characters), it is likely a non-answer and should be flagged.

In [ ]:
@output_guardrail
async def response_quality_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    output,
) -> GuardrailFunctionOutput:
    """Flag responses that are too short to be useful."""
    text = output.final_output if hasattr(output, "final_output") else str(output)
    is_too_short = len(text.strip()) < 20
    return GuardrailFunctionOutput(
        output_info=f"Response length: {len(text)} chars",
        tripwire_triggered=is_too_short,
    )

quality_agent = Agent(
    name="Quality-Checked IndiGo Agent",
    instructions="You are an IndiGo Airlines customer service agent. Give detailed, helpful answers.",
    model="gpt-4o-mini",
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
    output_guardrails=[response_quality_guardrail],
)

print("Quality-checked agent ready.")

In [ ]:
# Example 1: detailed question — agent gives a full answer, output guardrail passes
try:
    result = await Runner.run(quality_agent, "What is the excess baggage fee for domestic flights?")
    print("PASSED output guardrail:", result.final_output)
except OutputGuardrailTripwireTriggered as e:
    print("BLOCKED by output guardrail — response was too short.")

In [ ]:
# Example 2: combine both guardrails on one agent — input and output both checked
dual_guarded_agent = Agent(
    name="Dual Guarded IndiGo Agent",
    instructions="You are an IndiGo Airlines customer service agent. Give detailed, helpful answers.",
    model="gpt-4o-mini",
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
    input_guardrails=[airline_topic_guardrail],
    output_guardrails=[response_quality_guardrail],
)

# Valid question — both guardrails pass
try:
    result = await Runner.run(dual_guarded_agent, "Can I carry a musical instrument as cabin baggage?")
    print("PASSED both guardrails:", result.final_output)
except InputGuardrailTripwireTriggered:
    print("BLOCKED by input guardrail.")
except OutputGuardrailTripwireTriggered:
    print("BLOCKED by output guardrail.")

In [ ]:
# Off-topic question — input guardrail trips, agent never runs
try:
    result = await Runner.run(dual_guarded_agent, "What is a good recipe for biryani?")
    print("PASSED both guardrails:", result.final_output)
except InputGuardrailTripwireTriggered:
    print("BLOCKED by input guardrail — off-topic question rejected before agent ran.")
except OutputGuardrailTripwireTriggered:
    print("BLOCKED by output guardrail.")